In [0]:
%run ./silver_transformations

In [0]:
from pyspark.sql.functions import col

def get_last_watermark(source_name):
    watermark_table = "healthcare_catalog.metadata.silver_watermark"
    df=spark.read.table(watermark_table)\
    .filter(col('source_name')==source_name).orderBy(col('last_processed_ts').desc()).limit(1)

    if df.count()==0:
        return None
    return df.first()['last_processed_ts']


In [0]:
from pyspark.sql.functions import col
def read_incremental_bronze(bronze_table,last_watermark):
    if last_watermark:
        df=spark.read.table(bronze_table)\
            .filter(col('ingestion_ts')>last_watermark)
        return df
    else:
       df=spark.read.table(bronze_table)
       return df
            



In [0]:
def write_silver(df,silver_table):
    (
        df.write\
            .format('delta')\
                .mode('append')
                .saveAsTable(silver_table)
    )

In [0]:
def update_silver_watermark(source_name,silver_table,latest_ts):
    spark.createDataFrame(
        [
            {   'layer':'bronze to silver',
                'source_name':source_name,
                'last_processed_ts':latest_ts,
                'table_name':silver_table}
        ]
    )\
        .write\
            .format('delta')\
                .mode('append')\
                    .saveAsTable('healthcare_catalog.metadata.silver_watermark') 

In [0]:
from pyspark.sql.functions import trim,col
from pyspark.sql.types import StringType
def transform_data(df, source_name):
    print(f'genric transformation happening for source {source_name}')

    try:
        df=trim_string_columns(df)

        df =drop_null(df)
        
        df=drop_duplicates(df)

        print(f'transformation completed for source {source_name}')

        return df



    except Exception as e:
        print(f'error in generic transformation {e}')
